---
## This script generates daily heat index for the selected weather stations. (expanded edition)
#### Noel Siegert, 2/3/25
#### Heat index is computed from daily mean temperature (tmean) and mean dewpoint temp.
---

In [1]:
# imports
import os
import xarray as xr
import numpy as np
import netCDF4 
import glob
import pandas as pd
from datetime import datetime
import geopandas as gpd

/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [2]:
# interactive plotting stuff 
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

#import matplotlib.dates as mdates
%matplotlib inline
plt.rcParams['figure.figsize'] = 12, 6
#%config InlineBackend.figure_format = 'retina'

import cartopy
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

In [3]:
# script name
script = os.getcwd() + '/prep_daily_heatindex_expanded.ipynb'

In [4]:
def compute_heat_index(T, H):
    """
    This function uses the NWS's algorithm to compute heat index. 
    Source: Fig. 3 of this paper https://ehp.niehs.nih.gov/doi/10.1289/ehp.1206273
    This can be checked via this calculator: https://www.wpc.ncep.noaa.gov/html/heatindex.shtml
    (all the 'conditions' are verified to have worked)
    
    For whatever reason, they compute HI using fahrenheit. 
    
    Params: T (np.array) daily (mean) temperatures
            H (np.array) daily (mean) relative humidity
            
    Returns: HI (np.array) heat index, in degrees F (can convert to ˚C as well if needed)
    """
    
    # array to hold heat index values. 
    HI = np.zeros(shape=T.shape) * np.nan
    remaining_arr = np.ones(shape=T.shape) # this array will track the days that remain without a HI value. 
    
    # array to hold heat index values. 
    HI = np.zeros(shape=T.shape) * np.nan
    remaining_arr = np.ones(shape=T.shape) # this array will track the days that remain without a HI value. 
#    conditions_arr = np.zeros(shape=T.shape) # this array will track the conditions that applied to that HI calculation.

#    print('condition 0: {}'.format(np.sum(remaining_arr)))

    # CONDITION 1: if T <= 40F, HI = T
    HI[T<=40] = T[T <= 40]
    remaining_arr[T<=40] = 0
#    conditions_arr[T<=40] = 1
#    print('condition 1: {}'.format(np.sum(remaining_arr)))

    # otherwise, compute A:
    A = -10.3 + (1.1 * T) + (0.047 * H)

    # CONDITION 2: if A < 79F, HI = A
    HI[A<79] = A[A<79]
    remaining_arr[A<79] = 0
#    conditions_arr[A<79] = 2
#    print('condition 2: {}'.format(np.sum(remaining_arr)))

    # otherwise, compute B: (CAN I DO SOME ROUNDOFF HERE TO SAVE COMPUTING TIME...)
    B = -42.379 + (2.04901523*T) + (10.14333127*H) - (0.22475541*T*H) - (6.83783e-3*(T**2)) - (5.481717e-2*(H**2)) \
        + (1.22874e-3*(T**2)*H) + (8.5282e-4*T*(H**2)) - (1.99e-6*(T**2)*(H**2))

    # CONDITION 3: are H <= 13% and 80 <= T <= 112?
    cond3 = (H<=13)*(80<=T)*(T<=112)

    # if yes to cond. 3,
    cond3B = B - ((13-H)/4) * (((17-(np.abs(T - 95)))/17)**(1/2))
    HI[cond3] = cond3B[cond3]
    remaining_arr[cond3] = 0
#    conditions_arr[cond3] = 3
#    print('condition 3: {}'.format(np.sum(remaining_arr)))

    # CONDITION 4: Are H > 85% and 80 <= T <= 87?
    cond4 = (H>85)*(80<=T)*(T<=87)

    # if yes to cond. 4,
    cond4B = B + 0.02 * (H-85) * (87-T)
    HI[cond4] = cond4B[cond4]
    remaining_arr[cond4] = 0
#    conditions_arr[cond4] = 4
#    print('condition 4: {}'.format(np.sum(remaining_arr)))

    # otherwise, HI = B
    HI[remaining_arr.astype(bool)] = B[remaining_arr.astype(bool)]
#    conditions_arr[remaining_arr.astype(bool)] = 5
    
    # return the heat index
    return HI

In [5]:
# dataframe with the stations we are using
df = pd.read_csv('/home/nsiegert/projects/coastal_sst/data/hadisd_stations_using_Expanded.csv')
df = df.drop(['Unnamed: 0'], axis=1)

# convert df into geodataframe for ease of plotting
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(x=df.LON, y=df.LAT))
#gdf

In [6]:
# open station data
tx_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.tx.nc').Tx
tn_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.tn.nc').Tn
td_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.td.nc').Td

In [7]:
# generate daily average temp
tmean = (tx_da + tn_da) / 2

# convert tmean to Fahrenheit
tmean_f = (tmean * (9/5)) + 32

# compute rel. humidity
Td = td_da
rh = np.exp((17.625 * Td) / (243.04 + Td)) / np.exp((17.625 * tmean) / (243.04 + tmean)) * 100

In [8]:
# Compute HI and store in array
HI_arr = np.zeros(shape=tmean.shape) * np.nan

for stanum in range(len(gdf)):
    
    HI_arr[stanum] = compute_heat_index(tmean_f[stanum], rh[stanum])
    
    if stanum%100==0:
        print(stanum)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400


In [9]:
# convert HI to celsius, I guess
HI_c = (HI_arr - 32) * (5/9)

# put into dataArray
HI_da = xr.DataArray(HI_c, dims=tmean.dims, coords=tmean.coords)

In [10]:
# add attr's and save (will save rel. hum. too, why not)
now = datetime.now()

HI_da.attrs = {'script':script, 'timestamp':now.strftime("%Y-%m-%d %H:%M:%S"), 'units':'˚C', 'desc.':'Heat Index, Source: Fig. 3 of this paper https://ehp.niehs.nih.gov/doi/10.1289/ehp.1206273 This can be checked via this calculator: https://www.wpc.ncep.noaa.gov/html/heatindex.shtml'}
rh.attrs = {'script':script, 'timestamp':now.strftime("%Y-%m-%d %H:%M:%S"), 'units':'%', 'desc.':'Relative humidity.'}

In [12]:
# save
HI_da.to_dataset(name='HI').to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.HI.nc')
rh.to_dataset(name='rh').to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.rh.nc')